# GINO Evolution MLP SR Analysis

Tests the `mode="delta"` GINO MLP-decoder checkpoint with particle-position metrics and super-resolution field queries for `u`.

In [ ]:
from __future__ import annotations

import inspect
import json
import math
import os
import random
import sys
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    from neuralop.layers.gno_block import GNOBlock
except Exception as error:
    GNOBlock = None
    GNO_IMPORT_ERROR = error

SEED = int(os.environ.get("GINO_ANALYSIS_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _repo_paths() -> Tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    if (cwd / "FINAL").is_dir():
        return cwd, cwd / "FINAL"
    if cwd.name == "FINAL":
        return cwd.parent, cwd
    if cwd.parent.name == "FINAL":
        return cwd.parent.parent, cwd.parent
    return cwd, cwd / "FINAL"


def _safe_np_load(path: Path):
    try:
        return np.load(path, allow_pickle=True)
    except ModuleNotFoundError as error:
        if "numpy._core" not in str(error):
            raise
        import numpy.core as numpy_core
        sys.modules.setdefault("numpy._core", numpy_core)
        sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
        sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
        return np.load(path, allow_pickle=True)


def _torch_load(path: Path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

REPO_ROOT, FINAL_DIR = _repo_paths()
DATASET_PATH = Path(os.environ.get("GINO_EVOLUTION_DATASET", "")).expanduser()
if str(DATASET_PATH) in {"", "."}:
    DATASET_PATH = FINAL_DIR / "GNO evolution" / "processed_data" / "particle_evolution_dataset.npz"
CHECKPOINT_PATH = Path(os.environ.get("GINO_EVOLUTION_CHECKPOINT", "")).expanduser()
if str(CHECKPOINT_PATH) in {"", "."}:
    CHECKPOINT_PATH = FINAL_DIR / "result" / "task1_gino_evolution_mlp_sr" / "gino_evolution_mlp_sr_best_model.pt"
if not DATASET_PATH.is_absolute(): DATASET_PATH = (Path.cwd() / DATASET_PATH).resolve()
if not CHECKPOINT_PATH.is_absolute(): CHECKPOINT_PATH = (Path.cwd() / CHECKPOINT_PATH).resolve()
PLOTS_DIR = CHECKPOINT_PATH.parent / "analysis_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset   :", DATASET_PATH)
print("Checkpoint:", CHECKPOINT_PATH)
print("Plots     :", PLOTS_DIR)
print("Device    :", DEVICE)
if not DATASET_PATH.exists():
    raise FileNotFoundError(DATASET_PATH)
if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(CHECKPOINT_PATH)


In [ ]:
checkpoint = _torch_load(CHECKPOINT_PATH)
CFG = dict(checkpoint["config"])
feature_names = [str(x) for x in checkpoint["feature_names"]]
target_names = [str(x) for x in checkpoint["target_names"]]
feature_names_all_from_checkpoint = [str(x) for x in checkpoint["feature_names_all"]]

if target_names[:7] != ["dx", "dy", "dz", "dGamma_x", "dGamma_y", "dGamma_z", "dsigma"]:
    raise RuntimeError(f"Checkpoint is not a delta evolution checkpoint: {target_names}")
if target_names[7:10] not in (["u_x", "u_y", "u_z"], ["velocity_x", "velocity_y", "velocity_z"]):
    raise RuntimeError(f"Checkpoint does not contain particle u target channels: {target_names}")

dataset_file = _safe_np_load(DATASET_PATH)
feature_names_all = [str(x) for x in dataset_file["feature_names"].tolist()]
target_names_all = [str(x) for x in dataset_file["target_names"].tolist()]
active_input_feature_indices = [feature_names_all.index(name) for name in feature_names]
target_indices = [target_names_all.index(name) for name in target_names]
coord_feature_indices = [feature_names_all.index(name) for name in ("x", "y", "z")]
frame_contexts = list(dataset_file["pair_contexts"] if "pair_contexts" in dataset_file.files else dataset_file["frame_contexts"])
frame_ranges = list(dataset_file["pair_ranges"] if "pair_ranges" in dataset_file.files else dataset_file["frame_ranges"])
inputs_t = np.asarray(dataset_file["inputs_t"], dtype=np.float32)
targets_delta = np.asarray(dataset_file["targets_delta"], dtype=np.float32)
inputs_by_pair, targets_by_pair = [], []
for pair_range in frame_ranges:
    start, end = int(pair_range[3]), int(pair_range[4])
    inputs_by_pair.append(inputs_t[start:end])
    targets_by_pair.append(targets_delta[start:end][:, target_indices])
train_pair_ids = np.asarray(dataset_file["train_pair_ids"], dtype=np.int64)
val_pair_ids = np.asarray(dataset_file["val_pair_ids"], dtype=np.int64) if "val_pair_ids" in dataset_file.files else np.asarray([], dtype=np.int64)
test_pair_ids = np.asarray(dataset_file["test_pair_ids"], dtype=np.int64) if "test_pair_ids" in dataset_file.files else np.asarray([], dtype=np.int64)

input_mean = np.asarray(checkpoint["input_mean"], dtype=np.float32).reshape(-1)
input_std = np.maximum(np.asarray(checkpoint["input_std"], dtype=np.float32).reshape(-1), 1e-8)
target_mean = np.asarray(checkpoint["target_mean"], dtype=np.float32).reshape(-1)
target_std = np.maximum(np.asarray(checkpoint["target_std"], dtype=np.float32).reshape(-1), 1e-8)
coord_min = np.asarray(checkpoint["coord_min"], dtype=np.float32).reshape(3)
coord_span = np.maximum(np.asarray(checkpoint["coord_span"], dtype=np.float32).reshape(3), 1e-8)
target_mean_t = torch.tensor(target_mean, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
target_std_t = torch.tensor(target_std, dtype=torch.float32, device=DEVICE).view(1, 1, -1)

delta_position_slice = slice(0, 3)
delta_strength_slice = slice(3, 7)
particle_velocity_slice = slice(7, 10)

print("Checkpoint tag:", checkpoint.get("checkpoint_tag", "unknown"))
print("Features      :", feature_names)
print("Targets       :", target_names)
print("Splits        :", {"train": len(train_pair_ids), "val": len(val_pair_ids), "test": len(test_pair_ids)})


In [ ]:
def normalize_xyz(xyz: np.ndarray) -> np.ndarray:
    return np.clip((xyz.astype(np.float32) - coord_min[None, :]) / coord_span[None, :], 0.0, 1.0).astype(np.float32)

def sample_indices(n: int, cap: Optional[int], seed: int) -> np.ndarray:
    if cap is None or cap <= 0 or n <= int(cap):
        return np.arange(n, dtype=np.int64)
    rng = np.random.default_rng(int(seed))
    return np.sort(rng.choice(n, size=int(cap), replace=False)).astype(np.int64)

def denormalize_target(y):
    return y * target_std_t + target_mean_t

class EvolutionAnalysisDataset(Dataset):
    def __init__(self, pair_ids, split_name, max_input_particles=None, max_query_points=12000):
        self.pair_ids = np.asarray(pair_ids, dtype=np.int64)
        self.split_name = split_name
        self.max_input_particles = max_input_particles
        self.max_query_points = max_query_points
    def __len__(self): return int(len(self.pair_ids))
    def __getitem__(self, index):
        pair_id = int(self.pair_ids[int(index)])
        features_all = np.asarray(inputs_by_pair[pair_id], dtype=np.float32)
        targets_all = np.asarray(targets_by_pair[pair_id], dtype=np.float32)
        n = min(features_all.shape[0], targets_all.shape[0])
        in_idx = sample_indices(n, self.max_input_particles, SEED + pair_id)
        q_local = sample_indices(len(in_idx), self.max_query_points, SEED + 100000 + pair_id)
        q_idx = in_idx[q_local]
        input_features = features_all[in_idx]
        query_features = features_all[q_idx]
        x = (input_features[:, active_input_feature_indices] - input_mean[None, :]) / input_std[None, :]
        y = (targets_all[q_idx] - target_mean[None, :]) / target_std[None, :]
        return {"input_geom": torch.from_numpy(normalize_xyz(input_features[:, coord_feature_indices])).unsqueeze(0), "x": torch.from_numpy(np.clip(np.nan_to_num(x), -8.0, 8.0).astype(np.float32)).unsqueeze(0), "output_queries": torch.from_numpy(normalize_xyz(query_features[:, coord_feature_indices])).unsqueeze(0), "query_xyz_raw": torch.from_numpy(query_features[:, coord_feature_indices].astype(np.float32)).unsqueeze(0), "y": torch.from_numpy(np.nan_to_num(y).astype(np.float32)).unsqueeze(0), "pair_id": pair_id}

def make_gnoblock(in_channels, out_channels, radius):
    if GNOBlock is None:
        raise RuntimeError("neuralop.layers.gno_block.GNOBlock is not available") from GNO_IMPORT_ERROR
    kwargs = dict(in_channels=in_channels, out_channels=out_channels, coord_dim=3, radius=float(radius), transform_type="linear", reduction="mean", pos_embedding_type="transformer", pos_embedding_channels=12, channel_mlp_layers=[out_channels, out_channels, out_channels])
    accepted = set(inspect.signature(GNOBlock.__init__).parameters)
    if "use_torch_scatter_reduce" in accepted: kwargs["use_torch_scatter_reduce"] = False
    if "use_open3d_neighbor_search" in accepted: kwargs["use_open3d_neighbor_search"] = False
    return GNOBlock(**{k: v for k, v in kwargs.items() if k in accepted})

class LatentMLPDecoderGINO(nn.Module):
    def __init__(self, in_channels, out_channels, cfg):
        super().__init__()
        hidden = int(cfg["hidden_channels"])
        self.latent_res = int(cfg["latent_res"])
        self.lift = nn.Sequential(nn.Linear(in_channels, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        self.encoder = make_gnoblock(hidden, hidden, cfg["gno_radius"])
        mixer = []
        for _ in range(max(int(cfg["latent_mixer_layers"]), 1)):
            mixer += [nn.Conv3d(hidden, hidden, kernel_size=3, padding=1), nn.GELU()]
        self.latent_mixer = nn.Sequential(*mixer)
        width = int(cfg["mlp_hidden"])
        mlp = []
        for i in range(max(int(cfg["mlp_layers"]), 1)):
            mlp += [nn.Linear(hidden + 3 if i == 0 else width, width), nn.GELU()]
        mlp.append(nn.Linear(width, out_channels))
        self.decoder = nn.Sequential(*mlp)
    def forward(self, input_geom, latent_queries, output_queries, x):
        outs = []
        base_latent = latent_queries[0]
        r = self.latent_res
        for b in range(x.shape[0]):
            h = self.lift(x[b])
            latent = self.encoder(y=base_latent, x=input_geom[b], f_y=h)
            if latent.ndim == 3: latent = latent.squeeze(0)
            grid = latent.reshape(r, r, r, -1).permute(3, 0, 1, 2).unsqueeze(0)
            grid = self.latent_mixer(grid)
            q = output_queries[b].clamp(0.0, 1.0)
            sample_grid = (q * 2.0 - 1.0).view(1, -1, 1, 1, 3)
            sampled = torch.nn.functional.grid_sample(grid, sample_grid, align_corners=True, mode="bilinear")
            sampled = sampled.squeeze(0).squeeze(-1).squeeze(-1).transpose(0, 1)
            outs.append(self.decoder(torch.cat([sampled, q], dim=-1)))
        return torch.stack(outs, dim=0)

def make_latent_queries(res):
    line = torch.linspace(0.0, 1.0, int(res), dtype=torch.float32, device=DEVICE)
    xx, yy, zz = torch.meshgrid(line, line, line, indexing="ij")
    return torch.stack([xx, yy, zz], dim=-1).reshape(1, -1, 3)

LATENT_QUERIES = make_latent_queries(CFG["latent_res"])
model = LatentMLPDecoderGINO(len(feature_names), len(target_names), CFG).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Loaded model.")


In [ ]:
def move_batch(batch):
    return {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in batch.items()}

def relative_l2(pred, target, eps=1e-12):
    return torch.linalg.norm((pred - target).reshape(pred.shape[0], -1), dim=1) / torch.linalg.norm(target.reshape(target.shape[0], -1), dim=1).clamp_min(eps)

@torch.no_grad()
def evaluate(ds, max_items=12):
    if len(ds) == 0: return {"rel_l2": math.nan, "u_rel_l2": math.nan, "items": 0}
    rels, u_rels = [], []
    for i in range(min(len(ds), int(max_items))):
        batch = move_batch(ds[i])
        pred = model(batch["input_geom"], LATENT_QUERIES, batch["output_queries"], batch["x"]).float()
        pred_phys = denormalize_target(pred)
        y_phys = denormalize_target(batch["y"])
        rels.append(float(relative_l2(pred_phys, y_phys).mean().item()))
        u_rels.append(float(relative_l2(pred_phys[..., particle_velocity_slice], y_phys[..., particle_velocity_slice]).mean().item()))
    return {"rel_l2": float(np.mean(rels)), "u_rel_l2": float(np.mean(u_rels)), "items": len(rels)}

train_ds = EvolutionAnalysisDataset(train_pair_ids, "train", CFG.get("maximum_input_particles"), CFG.get("maximum_eval_query_points", 12000))
val_ds = EvolutionAnalysisDataset(val_pair_ids, "val", CFG.get("maximum_input_particles"), CFG.get("maximum_eval_query_points", 12000))
test_ds = EvolutionAnalysisDataset(test_pair_ids, "test", CFG.get("maximum_input_particles"), CFG.get("maximum_eval_query_points", 12000))
metrics = {"train": evaluate(train_ds), "val": evaluate(val_ds), "test": evaluate(test_ds)}
print(json.dumps(metrics, indent=2))


In [ ]:
@torch.no_grad()
def predict_u_on_field_grid(pair_id: int, grid_resolution: int = 96, y_value: Optional[float] = None):
    features_all = np.asarray(inputs_by_pair[int(pair_id)], dtype=np.float32)
    in_idx = sample_indices(features_all.shape[0], CFG.get("maximum_input_particles"), SEED + int(pair_id))
    input_features = features_all[in_idx]
    x = (input_features[:, active_input_feature_indices] - input_mean[None, :]) / input_std[None, :]
    x_line = np.linspace(coord_min[0], coord_min[0] + coord_span[0], grid_resolution, dtype=np.float32)
    z_line = np.linspace(coord_min[2], coord_min[2] + coord_span[2], grid_resolution, dtype=np.float32)
    xx, zz = np.meshgrid(x_line, z_line, indexing="xy")
    if y_value is None:
        y_value = float(np.median(features_all[:, coord_feature_indices[1]]))
    query_xyz = np.stack([xx, np.full_like(xx, y_value), zz], axis=-1).reshape(-1, 3)
    batch = {"input_geom": torch.from_numpy(normalize_xyz(input_features[:, coord_feature_indices])).unsqueeze(0).to(DEVICE), "x": torch.from_numpy(np.clip(np.nan_to_num(x), -8.0, 8.0).astype(np.float32)).unsqueeze(0).to(DEVICE), "output_queries": torch.from_numpy(normalize_xyz(query_xyz)).unsqueeze(0).to(DEVICE)}
    pred = model(batch["input_geom"], LATENT_QUERIES, batch["output_queries"], batch["x"]).float()
    u = denormalize_target(pred).squeeze(0).cpu().numpy()[:, particle_velocity_slice]
    return query_xyz, u.reshape(grid_resolution, grid_resolution, 3)

pair_source = test_pair_ids if len(test_pair_ids) else val_pair_ids if len(val_pair_ids) else train_pair_ids
pair_id = int(pair_source[0])
query_xyz, u_grid = predict_u_on_field_grid(pair_id, grid_resolution=int(CFG.get("sr_grid_resolution", 96)))
u_mag = np.linalg.norm(u_grid, axis=-1)
fig, ax = plt.subplots(figsize=(7.2, 5.5), constrained_layout=True)
im = ax.imshow(u_mag, origin="lower", extent=[query_xyz[:,0].min(), query_xyz[:,0].max(), query_xyz[:,2].min(), query_xyz[:,2].max()], aspect="auto")
ax.set_xlabel("x")
ax.set_ylabel("z")
ax.set_title(f"GINO SR field query |u|, pair_id={pair_id}")
plt.colorbar(im, ax=ax, label="|u|")
plot_path = PLOTS_DIR / "gino_sr_u_field_analysis.png"
fig.savefig(plot_path, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)
